In [1]:
#!/usr/bin/env python3
"""
DGH-XH: NDBC Gulf of Mexico — Significant Wave Height
=======================================================
Dataset : NOAA National Data Buoy Center (NDBC)
          9 moored buoys, Gulf of Mexico, 2019–2021
          Downloaded via NOAA ERDDAP API (built into this notebook)
Target  : WVHT = Significant Wave Height (metres)
Stations: 9 buoys (KNN k=4 graph)
Split   : test from 2021-06-01
Wind    : WDIR (degrees from N, met convention) + WSPD (m/s)
          → u/v via: u = -sin(θ)*spd, v = -cos(θ)*spd

Physical justification for wind-bearing gate:
  Ocean waves are generated and propagated in the wind direction.
  cos(θ_wind_going - β_edge) correctly encodes: "buoy j is downwind
  of buoy i, so wave energy generated at i will arrive at j."
  Identical gate formula to the AQI advection case.

Tail events: storm surge episodes (95th/99th pct wave height)
  These are the operationally critical events for maritime safety,
  directly analogous to hazardous AQI episodes.
"""

# ============================================================
# CELL 1 — IMPORTS
# ============================================================
import os, warnings, joblib, time
import numpy as np
import pandas as pd
import requests
from sklearn.pipeline import Pipeline
from sklearn.linear_model import HuberRegressor, Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    precision_recall_fscore_support, average_precision_score
)
import xgboost as xgb
from xgboost import XGBRegressor
from IPython.display import display
warnings.filterwarnings("ignore")

# ============================================================
# CELL 2 — CONFIG
# ============================================================
OUT_DIR     = "dghxh_ndbc_swh"
SPLIT_TIME  = pd.Timestamp("2021-06-01 00:00:00")
L           = 24
H_LIST      = [1, 3, 6, 12, 24]
TAU_LIST    = [0, 1, 2, 3, 4, 6]
VALID_FRAC  = 0.20
GRAPH_K     = 4
RUN_TUNED   = True
JSO_POP     = 6   # 48 evals (was 180): saves ~1.5h/horizon
JSO_ITERS   = 8   # reduced from 15
RANDOM_SEED = 1
TARGET_COL  = "wvht"
FEATURES    = ["wvht", "wtmp", "atmp", "pres", "u_wind", "v_wind"]

# ── Gulf of Mexico buoy coordinates ──────────────────────────
# All moored buoys with reliable multi-year records (verified from NDBC)
BUOY_COORDS = {
    "42001": (25.888, -89.658),  # Gulf of Mexico (central)
    "42002": (25.170, -94.420),  # Gulf of Mexico (western)
    "42003": (25.900, -85.900),  # Gulf of Mexico (eastern)
    "42019": (27.910, -95.350),  # Texas shelf
    "42020": (26.970, -96.690),  # South Texas
    "42035": (29.232, -94.413),  # Galveston offshore
    "42036": (28.500, -84.517),  # Florida Strait
    "42039": (28.790, -86.010),  # Florida shelf
    "42040": (29.212, -88.207),  # Mississippi canyon
}
BUOY_IDS = list(BUOY_COORDS.keys())

# ── Date range for download ───────────────────────────────────
START_DATE = "2019-01-01T00:00:00Z"
END_DATE   = "2021-12-31T23:59:00Z"

# ============================================================
# CELL 3 — NDBC DATA DOWNLOAD (direct from NOAA stdmet archive)
# ============================================================
# HOW THIS WORKS:
#   NDBC publishes one .txt.gz file per buoy per year at a stable URL:
#   https://www.ndbc.noaa.gov/data/historical/stdmet/{buoy_id}h{year}.txt.gz
#   Each file is ~50-200 KB, downloads in seconds, no API key needed.
#
# KAGGLE REQUIREMENT: Settings → Internet → ON  (must be enabled)
#
# NDBC stdmet column order:
#   YY MM DD hh mm WDIR WSPD GST WVHT DPD APD MWD PRES ATMP WTMP DEWP ...
#   WDIR = wind direction (degrees, met convention: direction FROM)
#   WSPD = wind speed (m/s)
#   WVHT = significant wave height (m)   ← our target
#   PRES = sea level pressure (hPa)
#   ATMP = air temperature (°C)
#   WTMP = water temperature (°C)
#   Missing values are encoded as 99, 999, 9999, 99.0, 999.0, 9999.0

DOWNLOAD_YEARS = [2019, 2020, 2021]

def download_ndbc_year(buoy_id, year, raw_dir, max_retries=3):
    """
    Download one year of NDBC stdmet data for a single buoy.
    Returns path to downloaded .txt.gz file, or None on failure.
    """
    url = (f"https://www.ndbc.noaa.gov/data/historical/stdmet/"
           f"{buoy_id}h{year}.txt.gz")
    out_path = os.path.join(raw_dir, f"{buoy_id}h{year}.txt.gz")

    if os.path.exists(out_path) and os.path.getsize(out_path) > 1000:
        return out_path   # already cached

    for attempt in range(max_retries):
        try:
            resp = requests.get(url, timeout=60)
            if resp.status_code == 200:
                with open(out_path, "wb") as f:
                    f.write(resp.content)
                return out_path
            else:
                print(f"    HTTP {resp.status_code} for {buoy_id}/{year}")
        except Exception as e:
            print(f"    Attempt {attempt+1} failed: {e}")
        time.sleep(2)
    return None

def parse_ndbc_file(gz_path, buoy_id):
    """
    Parse one NDBC stdmet .txt.gz file into a clean DataFrame.
    Handles both old format (no 'mm' minute column) and new format.
    Missing value sentinel: 99, 999, 9999 replaced with NaN.
    """
    import gzip
    with gzip.open(gz_path, 'rt') as f:
        lines = f.readlines()

    # Row 0 = header with # prefix, Row 1 = units row (also starts with #)
    # Detect if units row exists (NDBC added it around 2007)
    header_line = lines[0].lstrip('#').strip()
    cols = header_line.split()

    skip = 1
    if lines[1].startswith('#'):
        skip = 2   # skip units row

    data_lines = [l for l in lines[skip:] if not l.startswith('#')]
    from io import StringIO
    df = pd.read_csv(StringIO(''.join(data_lines)),
                     sep=r'\s+', header=None, names=cols,
                     na_values=['99','999','9999','99.0','999.0','9999.0','MM'])

    # Build timestamp — handle both 4-digit and 2-digit year
    yr_col = 'YY' if 'YY' in df.columns else 'YYYY'
    df[yr_col] = df[yr_col].astype(int)
    if df[yr_col].max() < 100:
        df[yr_col] = df[yr_col].apply(lambda y: y+2000 if y<50 else y+1900)

    mm_col = 'mm' if 'mm' in df.columns else None  # minute column
    try:
        if mm_col:
            df['timestamp'] = pd.to_datetime(
                dict(year=df[yr_col], month=df['MM'], day=df['DD'],
                     hour=df['hh'], minute=df[mm_col]))
        else:
            df['timestamp'] = pd.to_datetime(
                dict(year=df[yr_col], month=df['MM'], day=df['DD'],
                     hour=df['hh']))
    except Exception:
        return None

    df = df.sort_values('timestamp').drop_duplicates('timestamp')

    # Map to standard column names
    col_map = {'WDIR':'wdir','WSPD':'wspd','WVHT':'wvht',
               'ATMP':'atmp','WTMP':'wtmp','PRES':'pres'}
    df = df.rename(columns=col_map)

    for col in ['wdir','wspd','wvht','atmp','wtmp','pres']:
        if col not in df.columns:
            df[col] = np.nan
        else:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Convert met wind direction + speed to u/v components
    # Met convention: direction wind comes FROM, clockwise from N
    # u = -sin(θ)*spd  (eastward motion),  v = -cos(θ)*spd  (northward motion)
    theta = np.radians(df['wdir'].fillna(0).to_numpy())
    spd   = df['wspd'].fillna(0).to_numpy()
    df['u_wind'] = (-np.sin(theta) * spd).astype(np.float32)
    df['v_wind'] = (-np.cos(theta) * spd).astype(np.float32)

    df['station_id'] = buoy_id
    df['lat'] = BUOY_COORDS[buoy_id][0]
    df['lon'] = BUOY_COORDS[buoy_id][1]

    return df[['timestamp','station_id','lat','lon',
               'wvht','wtmp','atmp','pres','u_wind','v_wind']]

def build_ndbc_dataset():
    """
    Download NDBC stdmet files for all buoys and years,
    parse, concatenate, resample to hourly, and cache as parquet.
    Requires Kaggle Internet = ON.
    """
    os.makedirs(OUT_DIR, exist_ok=True)
    raw_dir    = os.path.join(OUT_DIR, "raw_gz")
    cache_path = os.path.join(OUT_DIR, "ndbc_raw_cache.parquet")
    os.makedirs(raw_dir, exist_ok=True)

    if os.path.exists(cache_path):
        print(f"Loading from cache: {cache_path}")
        return pd.read_parquet(cache_path)

    print("Downloading NDBC stdmet files from NOAA "
          "(requires Kaggle Internet = ON)...")
    all_dfs = []
    for buoy_id in BUOY_IDS:
        buoy_dfs = []
        for year in DOWNLOAD_YEARS:
            print(f"  {buoy_id} {year}...", end=" ", flush=True)
            gz_path = download_ndbc_year(buoy_id, year, raw_dir)
            if gz_path is None:
                print("SKIP")
                continue
            parsed = parse_ndbc_file(gz_path, buoy_id)
            if parsed is None or len(parsed) == 0:
                print("PARSE ERROR")
                continue
            buoy_dfs.append(parsed)
            print(f"OK ({len(parsed)} rows)")

        if buoy_dfs:
            # Concatenate years and resample to strict hourly
            buoy_df = (pd.concat(buoy_dfs, ignore_index=True)
                         .sort_values("timestamp")
                         .drop_duplicates("timestamp")
                         .set_index("timestamp")
                         .resample("h")
                         .first()
                         .reset_index())
            # Restore station metadata after resample
            buoy_df["station_id"] = buoy_id
            buoy_df["lat"] = BUOY_COORDS[buoy_id][0]
            buoy_df["lon"] = BUOY_COORDS[buoy_id][1]
            all_dfs.append(buoy_df)
            print(f"  → {buoy_id}: {len(buoy_df)} hourly rows total")
        else:
            print(f"  → {buoy_id}: NO DATA — skipping this buoy")

    if not all_dfs:
        raise RuntimeError(
            "No buoy data loaded.\n"
            "Check: 1) Kaggle Internet is ON  "
            "2) NOAA NDBC server is reachable  "
            "3) Buoy IDs are valid")

    df_all = (pd.concat(all_dfs, ignore_index=True)
                .sort_values(["timestamp","station_id"])
                .reset_index(drop=True))
    df_all.to_parquet(cache_path, index=False)
    print(f"\nSaved cache: {cache_path}")
    print(f"Total: {len(df_all):,} rows, {df_all.station_id.nunique()} buoys")
    print(f"Period: {df_all.timestamp.min()} → {df_all.timestamp.max()}")
    return df_all

# ============================================================
# CELL 4 — ALL SHARED HELPER FUNCTIONS (identical to other notebooks)
# ============================================================
def haversine_km_vec(lat1,lon1,lat2,lon2):
    R=6371.0; lat1,lon1,lat2,lon2=map(np.radians,[lat1,lon1,lat2,lon2])
    dlat=lat2-lat1; dlon=lon2-lon1
    a=np.sin(dlat/2)**2+np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2.0*R*np.arcsin(np.sqrt(a))

def bearing_radians(lat1,lon1,lat2,lon2):
    lat1,lon1,lat2,lon2=map(np.radians,[lat1,lon1,lat2,lon2]); dlon=lon2-lon1
    y=np.sin(dlon)*np.cos(lat2); x=np.cos(lat1)*np.sin(lat2)-np.sin(lat1)*np.cos(lat2)*np.cos(dlon)
    return np.arctan2(y,x)

def flatten_window_per_node(X):
    S,Lx,N,F=X.shape; return X.transpose(0,2,1,3).reshape(S*N,Lx*F)

def compute_tail_metrics(y_true,y_pred,percentiles=(90,95,99)):
    rows=[]
    for p in percentiles:
        thr=np.percentile(y_true,p); idx=y_true>=thr
        if idx.sum()==0: rows.append((p,thr,np.nan,np.nan,0))
        else: rows.append((p,thr,mean_absolute_error(y_true[idx],y_pred[idx]),
            np.sqrt(mean_squared_error(y_true[idx],y_pred[idx])),int(idx.sum())))
    return rows

def compute_mfb_nmse(yt,yp):
    yt=np.asarray(yt,dtype=float); yp=np.asarray(yp,dtype=float)
    return np.mean(2*(yp-yt)/(yp+yt+1e-8)), np.sum((yp-yt)**2)/(np.sum(yp*yt)+1e-8)

def build_fill_values_from_train_timeline(X):
    fv=np.nanmedian(X,axis=0); gf=np.nanmedian(X.reshape(-1,X.shape[-1]),axis=0)
    gf=np.where(np.isnan(gf),0,gf)
    for n in range(fv.shape[0]):
        for f in range(fv.shape[1]):
            if np.isnan(fv[n,f]): fv[n,f]=gf[f]
    return np.where(np.isnan(fv),0,fv).astype(np.float32)

def impute_windows(X,fv):
    X_imp=X.copy().astype(np.float32); S,Lx,N,F=X_imp.shape
    for s in range(S):
        for n in range(N):
            for f in range(F):
                ser=pd.Series(X_imp[s,:,n,f],dtype="float32").ffill().bfill()
                arr=ser.to_numpy(dtype=np.float32)
                if np.isnan(arr).any(): arr=np.where(np.isnan(arr),fv[n,f],arr)
                X_imp[s,:,n,f]=arr
    return X_imp

def build_nodes_from_coords(station_ids,coord_dict):
    return pd.DataFrame([{"station_id":sid,"lat":coord_dict[sid][0],
                           "lon":coord_dict[sid][1],"node_id":i}
                          for i,sid in enumerate(station_ids)])

def build_edges_from_nodes(nodes_df,k=4):
    coords=nodes_df[["lat","lon"]].to_numpy(dtype=float); N=len(coords)
    D=np.zeros((N,N))
    for i in range(N): D[i,:]=haversine_km_vec(coords[i,0],coords[i,1],coords[:,0],coords[:,1])
    sigma=np.median(D[D>0]); edges=[]
    for i in range(N):
        for j in np.argsort(D[i])[1:min(k+1,N)]:
            edges.append((i,j,float(np.exp(-(D[i,j]**2)/(2*sigma**2))),float(D[i,j])))
    return pd.DataFrame(edges,columns=["src","dst","w_dist","dist_km"])

def build_static_adj(nodes_df,k=4):
    coords=nodes_df[["lat","lon"]].to_numpy(dtype=float); N=len(coords)
    D=np.zeros((N,N))
    for i in range(N): D[i,:]=haversine_km_vec(coords[i,0],coords[i,1],coords[:,0],coords[:,1])
    sigma=np.median(D[D>0]); A=np.zeros((N,N),dtype=np.float32)
    for i in range(N):
        for j in np.argsort(D[i])[1:min(k+1,N)]:
            A[i,j]=np.exp(-(D[i,j]**2)/(2*sigma**2))
    rs=A.sum(axis=1,keepdims=True); return np.divide(A,rs,out=np.zeros_like(A),where=rs>0)

def build_dynamic_adj(u_t,v_t,ctx,alpha=4.0,eps=0.05):
    theta_w=np.arctan2(v_t[ctx["src"]],u_t[ctx["src"]])
    gate=eps+(1-eps)/(1+np.exp(-alpha*np.cos(theta_w-ctx["edge_bearing"])))
    w=ctx["w_dist"]*gate; A=np.zeros((ctx["N"],ctx["N"]),dtype=np.float32)
    A[ctx["src"],ctx["dst"]]=w.astype(np.float32)
    rs=A.sum(axis=1,keepdims=True); return np.divide(A,rs,out=np.zeros_like(A),where=rs>0)

def make_graph_features_dynamic(X,ctx,tau=0,alpha=4.0,eps=0.05):
    S,Lx,N,F=X.shape; Z=np.zeros((S,N,2*F),dtype=np.float32)
    for s in range(S):
        x_now=X[s,-1]; x_lag=X[s,-1-tau] if tau>0 else x_now
        A=build_dynamic_adj(x_now[:,ctx["u_idx"]],x_now[:,ctx["v_idx"]],ctx,alpha,eps)
        Z[s]=np.concatenate([x_now,A@x_lag],axis=-1)
    return Z

def make_graph_features_static(X,A_static,tau=0):
    S,Lx,N,F=X.shape; Z=np.zeros((S,N,2*F),dtype=np.float32)
    for s in range(S):
        x_now=X[s,-1]; x_lag=X[s,-1-tau] if tau>0 else x_now
        Z[s]=np.concatenate([x_now,A_static@x_lag],axis=-1)
    return Z

def summarize_regression(yt,yp):
    return {"MAE":float(mean_absolute_error(yt,yp)),"RMSE":float(np.sqrt(mean_squared_error(yt,yp))),"R2":float(r2_score(yt,yp))}

def train_val_split_time_order(X,Y,M,times,frac=0.2):
    n=X.shape[0]; v=max(1,int(np.ceil(frac*n)))
    return ({"X":X[:n-v],"Y":Y[:n-v],"M":M[:n-v],"times":times[:n-v]},
            {"X":X[n-v:],"Y":Y[n-v:],"M":M[n-v:],"times":times[n-v:]})

def fit_flat_regressor(model,Xtr,Ytr,Mtr,Xte,Yte,Mte):
    Xt2=flatten_window_per_node(Xtr); Xe2=flatten_window_per_node(Xte)
    yt=Ytr.reshape(-1); ye=Yte.reshape(-1); mt=(Mtr.reshape(-1)>0.5); me=(Mte.reshape(-1)>0.5)
    model.fit(Xt2[mt],yt[mt]); return {"y_true":ye[me],"y_pred":model.predict(Xe2)[me],"model":model}

def fit_static_graph_regressor(model,Xtr,Ytr,Mtr,Xte,Yte,Mte,A_static,tau):
    Zt=make_graph_features_static(Xtr,A_static,tau); Ze=make_graph_features_static(Xte,A_static,tau)
    yt=Ytr.reshape(-1); ye=Yte.reshape(-1); mt=(Mtr.reshape(-1)>0.5); me=(Mte.reshape(-1)>0.5)
    Xt2=Zt.reshape(-1,Zt.shape[-1]); Xe2=Ze.reshape(-1,Ze.shape[-1])
    model.fit(Xt2[mt],yt[mt]); return {"y_true":ye[me],"y_pred":model.predict(Xe2)[me],"model":model}

def fit_huber_graph(Xtr,Ytr,Mtr,Xte,Yte,Mte,ctx,tau=0,alpha=4.0,eps=0.05,huber_alpha=1e-4):
    Zt=make_graph_features_dynamic(Xtr,ctx,tau,alpha,eps); Ze=make_graph_features_dynamic(Xte,ctx,tau,alpha,eps)
    yt=Ytr.reshape(-1); ye=Yte.reshape(-1); mt=(Mtr.reshape(-1)>0.5); me=(Mte.reshape(-1)>0.5)
    Xt2=Zt.reshape(-1,Zt.shape[-1]); Xe2=Ze.reshape(-1,Ze.shape[-1])
    huber=Pipeline([("sc",StandardScaler()),("h",HuberRegressor(epsilon=1.35,alpha=huber_alpha,max_iter=500))])
    huber.fit(Xt2[mt],yt[mt]); ptr=huber.predict(Xt2); pte=huber.predict(Xe2)
    return {"y_true":ye[me],"y_pred":pte[me],"pred_train_all":ptr,"pred_test_all":pte,
            "y_train_all":yt,"y_test_all":ye,"mask_train":mt,"mask_test":me,"model":huber}

def fit_huber_hybrid(Xtr,Ytr,Mtr,Xte,Yte,Mte,ctx,tau=0,alpha=4.0,eps=0.05,huber_alpha=1e-4,
                     xgb_params=None,num_boost_round=400,seed=42,w_mid=2.0,w_danger=5.0,w_tail=10.0):
    base=fit_huber_graph(Xtr,Ytr,Mtr,Xte,Yte,Mte,ctx,tau,alpha,eps,huber_alpha)
    yt=base["y_train_all"]; mt=base["mask_train"]; me=base["mask_test"]
    res=yt-base["pred_train_all"]
    Xg_tr=np.asarray(flatten_window_per_node(Xtr),dtype=np.float32)
    Xg_te=np.asarray(flatten_window_per_node(Xte),dtype=np.float32)
    t90,t95,t99=np.percentile(yt[mt],[90,95,99])
    w=np.ones_like(yt[mt],dtype=np.float32)
    w[yt[mt]>=t90]=w_mid; w[yt[mt]>=t95]=w_danger; w[yt[mt]>=t99]=w_tail
    dtrain=xgb.DMatrix(Xg_tr[mt],label=res[mt],weight=w)
    if xgb_params is None:
        xgb_params={"objective":"reg:pseudohubererror","max_depth":6,"eta":0.05,
                    "subsample":0.80,"colsample_bytree":0.80,"lambda":1.0,"tree_method":"hist","seed":seed,"verbosity":0}
    booster=xgb.train(xgb_params,dtrain,num_boost_round=int(num_boost_round))
    res_te=booster.predict(xgb.DMatrix(Xg_te)); final=base["pred_test_all"].copy(); final[me]=final[me]+res_te[me]
    return {"y_true":base["y_test_all"][me],"y_pred_graph":base["pred_test_all"][me],"y_pred_final":final[me],
            "gmodel":base["model"],"hmodel":booster}

def choose_best_tau_dynamic(tr,va,ctx,tau_list):
    best_tau,best_mae=None,np.inf; rows=[]
    for tau in tau_list:
        if tau>=tr["X"].shape[1]: continue
        res=fit_huber_graph(tr["X"],tr["Y"],tr["M"],va["X"],va["Y"],va["M"],ctx,tau=tau)
        mae=mean_absolute_error(res["y_true"],res["y_pred"]); rows.append({"tau":tau,"val_MAE":mae})
        if mae<best_mae: best_mae=mae; best_tau=tau
    return best_tau, pd.DataFrame(rows)

def choose_best_tau_static(tr,va,A_static,tau_list):
    best_tau,best_mae=None,np.inf; rows=[]
    for tau in tau_list:
        if tau>=tr["X"].shape[1]: continue
        m=Pipeline([("sc",StandardScaler()),("h",HuberRegressor(epsilon=1.35,max_iter=500))])
        res=fit_static_graph_regressor(m,tr["X"],tr["Y"],tr["M"],va["X"],va["Y"],va["M"],A_static,tau)
        mae=mean_absolute_error(res["y_true"],res["y_pred"]); rows.append({"tau":tau,"val_MAE":mae})
        if mae<best_mae: best_mae=mae; best_tau=tau
    return best_tau, pd.DataFrame(rows)

def eval_hybrid_params(params,tau,tr,va,ctx):
    al,ep,lha,md,eta,sub,col,lam,nr,wm,wd_w,wt=params
    hybrid=fit_huber_hybrid(tr["X"],tr["Y"],tr["M"],va["X"],va["Y"],va["M"],ctx,tau=tau,
        alpha=float(al),eps=float(ep),huber_alpha=10**float(lha),
        xgb_params={"objective":"reg:pseudohubererror","max_depth":int(round(md)),"eta":float(eta),
            "subsample":float(sub),"colsample_bytree":float(col),"lambda":float(lam),
            "tree_method":"hist","seed":RANDOM_SEED,"verbosity":0},
        num_boost_round=int(round(nr)),seed=RANDOM_SEED,w_mid=float(wm),w_danger=float(wd_w),w_tail=float(wt))
    mae=mean_absolute_error(hybrid["y_true"],hybrid["y_pred_final"])
    idx=hybrid["y_true"]>=np.percentile(hybrid["y_true"],99)
    if idx.sum()>0:
        mfb,_=compute_mfb_nmse(hybrid["y_true"][idx],hybrid["y_pred_final"][idx])
        return mae+10*abs(mfb)
    return mae

def jso_optimize(tr,va,ctx,tau,bounds,n_pop=12,iters=15,seed=1):
    np.random.seed(seed); dim=bounds.shape[0]
    pop=bounds[:,0]+np.random.rand(n_pop,dim)*(bounds[:,1]-bounds[:,0])
    fit=np.array([eval_hybrid_params(p,tau,tr,va,ctx) for p in pop])
    best_p=pop[np.argmin(fit)].copy(); best_f=float(fit.min())
    for it in range(iters):
        c=1-it/max(iters,1); new_pop=pop.copy()
        for i in range(n_pop):
            if np.random.rand()<0.5:
                cand=pop[i]+np.random.randn(dim)*c*0.1+c*(best_p-pop[i])*np.random.rand(dim)
            else:
                j=np.random.randint(0,n_pop); cand=pop[i]+(pop[j]-pop[i])*(np.random.rand(dim)-0.5)*c
            new_pop[i]=np.clip(cand,bounds[:,0],bounds[:,1])
        nf=np.array([eval_hybrid_params(p,tau,tr,va,ctx) for p in new_pop])
        better=nf<fit; pop[better]=new_pop[better]; fit[better]=nf[better]
        if fit.min()<best_f: best_f=float(fit.min()); best_p=pop[np.argmin(fit)].copy()
        print(f"  iter {it+1:02d}/{iters} best={best_f:.4f}")
    return best_p, best_f

def add_result_row(rows,tail_rows,H,split,model,y_true,y_pred,extra=None):
    s=summarize_regression(y_true,y_pred)
    row={"H":H,"split":split,"model":model,"MAE":s["MAE"],"RMSE":s["RMSE"],"R2":s["R2"],"n_test":len(y_true)}
    if extra: row.update(extra)
    rows.append(row)
    for p,thr,mae,rmse,n_tail in compute_tail_metrics(y_true,y_pred):
        tr={"H":H,"split":split,"model":model,"percentile":p,"threshold":float(thr),
            "tail_MAE":float(mae) if pd.notna(mae) else np.nan,
            "tail_RMSE":float(rmse) if pd.notna(rmse) else np.nan,"n_tail":int(n_tail)}
        if extra: tr.update(extra)
        tail_rows.append(tr)

# ============================================================
# CELL 5 — BUILD TENSOR
# ============================================================
os.makedirs(OUT_DIR, exist_ok=True)
df = build_ndbc_dataset()

# Keep only buoys that actually downloaded
station_ids = [bid for bid in BUOY_IDS if bid in df["station_id"].unique()]
N = len(station_ids)
print(f"Active buoys: {N}  ({station_ids})")
assert N >= 5, f"Too few buoys ({N}); need at least 5 for k=4 graph"

u_idx = FEATURES.index("u_wind"); v_idx = FEATURES.index("v_wind")

all_times = pd.date_range(df["timestamp"].min(), df["timestamp"].max(), freq="h")
base = pd.MultiIndex.from_product([all_times, station_ids],
                                   names=["timestamp","station_id"]).to_frame(index=False)
aligned = base.merge(df[["timestamp","station_id"]+FEATURES],
                     on=["timestamp","station_id"], how="left")

X_feat = []
for feat in FEATURES:
    mat = aligned.pivot(index="timestamp", columns="station_id",
                        values=feat).reindex(all_times)[station_ids]
    X_feat.append(mat.to_numpy(dtype=np.float32))
X_all_raw = np.stack(X_feat, axis=-1)

target_raw = aligned.pivot(index="timestamp", columns="station_id",
                            values=TARGET_COL).reindex(all_times)[station_ids].to_numpy(dtype=np.float32)
Y_mask_full = (~np.isnan(target_raw)).astype(np.float32)
fill_values = build_fill_values_from_train_timeline(X_all_raw[all_times < SPLIT_TIME])

print(f"Tensor {X_all_raw.shape}  WVHT coverage: {Y_mask_full.mean()*100:.1f}%")
print(f"WVHT range: {np.nanmin(target_raw):.2f} – {np.nanmax(target_raw):.2f} m")
print(f"WVHT 99th pct: {np.nanpercentile(target_raw,99):.2f} m (storm threshold)")

# ============================================================
# CELL 6 — BUILD GRAPH
# ============================================================
nodes    = build_nodes_from_coords(station_ids, BUOY_COORDS)
edges_df = build_edges_from_nodes(nodes, k=min(GRAPH_K, N-1))
A_static = build_static_adj(nodes, k=min(GRAPH_K, N-1))

src = edges_df["src"].to_numpy(dtype=int)
dst = edges_df["dst"].to_numpy(dtype=int)
w_dist = edges_df["w_dist"].to_numpy(dtype=np.float32)
edge_bearing = bearing_radians(
    nodes.loc[src,"lat"].to_numpy(), nodes.loc[src,"lon"].to_numpy(),
    nodes.loc[dst,"lat"].to_numpy(), nodes.loc[dst,"lon"].to_numpy())

graph_ctx = {"src":src,"dst":dst,"w_dist":w_dist,"edge_bearing":edge_bearing,
             "u_idx":u_idx,"v_idx":v_idx,"N":N}

# ============================================================
# CELL 7 — WINDOWING
# ============================================================
def build_windows(X_raw,Y_raw,Y_mask,all_times,split_time,L,H_list):
    n_times=len(all_times); data={}
    for H in H_list:
        X_wins,Y_wins,M_wins,ttimes=[],[],[],[]
        for t in range(L, n_times-H):
            X_wins.append(X_raw[t-L:t]); Y_wins.append(Y_raw[t+H])
            M_wins.append(Y_mask[t+H]); ttimes.append(all_times[t+H])
        X_arr=np.stack(X_wins); Y_arr=np.stack(Y_wins)
        M_arr=np.stack(M_wins); t_arr=np.array(ttimes)
        test_mask=t_arr>=split_time; train_mask=~test_mask
        data[H]={"X_train_imp":impute_windows(X_arr[train_mask],fill_values),
                 "X_test_imp": impute_windows(X_arr[test_mask], fill_values),
                 "Y_train":Y_arr[train_mask],"Y_test":Y_arr[test_mask],
                 "M_train":M_arr[train_mask],"M_test":M_arr[test_mask],
                 "target_times_train":t_arr[train_mask],"target_times_test":t_arr[test_mask]}
        print(f"H={H}h  train={train_mask.sum()}  test={test_mask.sum()}")
    return data

print("Building windows...")
all_data = build_windows(X_all_raw, target_raw, Y_mask_full, all_times, SPLIT_TIME, L, H_LIST)

bounds = np.array([[1.0,8.0],[0.01,0.20],[-5.0,-2.0],[3.0,8.0],[0.02,0.15],
                   [0.60,1.00],[0.60,1.00],[0.10,5.0],[200.0,600.0],
                   [1.5,4.0],[3.0,8.0],[6.0,20.0]])

# ============================================================
# CELL 8 — MAIN TRAINING LOOP (identical)
# ============================================================
results_rows, tail_rows, tau_rows = [], [], []
all_models = {}

for H in H_LIST:
    print(f"\n{'='*50}\nH={H}h  [NDBC WVHT]\n{'='*50}")
    pack=all_data[H]
    Xtr=pack["X_train_imp"]; Ytr=pack["Y_train"]; Mtr=pack["M_train"]
    Xte=pack["X_test_imp"];  Yte=pack["Y_test"];  Mte=pack["M_test"]
    tr,va=train_val_split_time_order(Xtr,Ytr,Mtr,pack["target_times_train"],frac=VALID_FRAC)
    btd,_=choose_best_tau_dynamic(tr,va,graph_ctx,TAU_LIST)
    bts,_=choose_best_tau_static(tr,va,A_static,TAU_LIST)
    all_models[H]={"tau_dyn":btd,"tau_static":bts}

    for name,model in [
        ("Ridge",Pipeline([("sc",StandardScaler()),("r",Ridge(alpha=10.0))])),
        ("Huber",Pipeline([("sc",StandardScaler()),("h",HuberRegressor(epsilon=1.35,max_iter=1000))])),
        ("RF",RandomForestRegressor(n_estimators=100,max_depth=8,n_jobs=-1,random_state=RANDOM_SEED)),  # reduced for speed),
        ("XGB",XGBRegressor(n_estimators=200,max_depth=8,learning_rate=0.07,subsample=0.9,
                             colsample_bytree=0.8,tree_method="hist",n_jobs=-1,random_state=RANDOM_SEED)),
    ]:
        res=fit_flat_regressor(model,Xtr,Ytr,Mtr,Xte,Yte,Mte)
        add_result_row(results_rows,tail_rows,H,"test",name,res["y_true"],res["y_pred"])

    for name,model in [
        ("SG-Ridge",Pipeline([("sc",StandardScaler()),("r",Ridge(alpha=10.0))])),
        ("SG-Huber",Pipeline([("sc",StandardScaler()),("h",HuberRegressor(epsilon=1.35,max_iter=1000))])),
    ]:
        res=fit_static_graph_regressor(model,Xtr,Ytr,Mtr,Xte,Yte,Mte,A_static,bts)
        add_result_row(results_rows,tail_rows,H,"test",name,res["y_true"],res["y_pred"],{"tau":bts})

    res=fit_huber_graph(Xtr,Ytr,Mtr,Xte,Yte,Mte,graph_ctx,tau=btd)
    add_result_row(results_rows,tail_rows,H,"test","Huber-Graph",res["y_true"],res["y_pred"],{"tau":btd})
    res=fit_huber_hybrid(Xtr,Ytr,Mtr,Xte,Yte,Mte,graph_ctx,tau=btd,num_boost_round=400,seed=RANDOM_SEED)
    add_result_row(results_rows,tail_rows,H,"test","Hybrid",res["y_true"],res["y_pred_final"],{"tau":btd})

    if RUN_TUNED:
        best_p,best_obj=jso_optimize(tr,va,graph_ctx,btd,bounds,JSO_POP,JSO_ITERS,RANDOM_SEED)
        al,ep,lha,md,eta,sub,col,lam,nr,wm,wd_w,wt=best_p; ha=10**float(lha)
        res=fit_huber_graph(Xtr,Ytr,Mtr,Xte,Yte,Mte,graph_ctx,tau=btd,alpha=float(al),eps=float(ep),huber_alpha=ha)
        add_result_row(results_rows,tail_rows,H,"test","Tuned-Huber-Graph",res["y_true"],res["y_pred"],{"tau":btd,"val_obj":float(best_obj)})
        res=fit_huber_hybrid(Xtr,Ytr,Mtr,Xte,Yte,Mte,graph_ctx,tau=btd,alpha=float(al),eps=float(ep),huber_alpha=ha,
            xgb_params={"objective":"reg:pseudohubererror","max_depth":int(round(md)),"eta":float(eta),
                "subsample":float(sub),"colsample_bytree":float(col),"lambda":float(lam),
                "tree_method":"hist","seed":RANDOM_SEED,"verbosity":0},
            num_boost_round=int(round(nr)),seed=RANDOM_SEED,w_mid=float(wm),w_danger=float(wd_w),w_tail=float(wt))
        add_result_row(results_rows,tail_rows,H,"test","Tuned-Hybrid",res["y_true"],res["y_pred_final"],{"tau":btd,"val_obj":float(best_obj)})
    print(f"H={H} done")

# ============================================================
# CELL 9 — SAVE
# ============================================================
results_df=pd.DataFrame(results_rows).sort_values(["H","MAE"]).reset_index(drop=True)
tail_df=pd.DataFrame(tail_rows).sort_values(["H","model","percentile"]).reset_index(drop=True)
results_df.to_csv(f"{OUT_DIR}/results_main.csv",index=False)
tail_df.to_csv(f"{OUT_DIR}/results_tail.csv",index=False)
joblib.dump(all_models,f"{OUT_DIR}/models.pkl")
print("\n=== NDBC WAVE HEIGHT RESULTS ==="); display(results_df)
print("\n=== STORM EVENTS (99th pct tail MAE) ===")
display(tail_df[tail_df.percentile==99][["H","model","tail_MAE","threshold"]].dropna())


  42001 2019... OK (29802 rows)
  42001 2020... OK (37579 rows)
  42001 2021... OK (13645 rows)
  → 42001: 18854 hourly rows total
  42002 2019... OK (17211 rows)
  42002 2020... OK (52273 rows)
  42002 2021... OK (51489 rows)
  → 42002: 26304 hourly rows total
  42003 2019... OK (36977 rows)
  42003 2020... OK (52250 rows)
  42003 2021... OK (47046 rows)
  → 42003: 25562 hourly rows total
  42019 2019... OK (34213 rows)
  42019 2020... OK (46506 rows)
  42019 2021... OK (51564 rows)
  → 42019: 26304 hourly rows total
  42020 2019... OK (52129 rows)
  42020 2020... OK (33062 rows)
  42020 2021... OK (22875 rows)
  → 42020: 26304 hourly rows total
  42035 2019... OK (52113 rows)
  42035 2020... OK (52136 rows)
  42035 2021... OK (51518 rows)
  → 42035: 26304 hourly rows total
  42036 2019... OK (26943 rows)
  42036 2020... OK (52110 rows)
  42036 2021... OK (51496 rows)
  → 42036: 22038 hourly rows total
  42039 2019... OK (26901 rows)
  42039 2020... OK (52273 rows)
  42039 2021... OK 

,H,split,model,MAE,RMSE,R2,n_test,tau,val_obj
0,1,test,Hybrid,0.072306,0.121437,0.956263,36224,0.0,NaN
1,1,test,XGB,0.073019,0.132772,0.947716,36224,NaN,NaN
2,1,test,Huber,0.073300,0.122612,0.955413,36224,NaN,NaN
3,1,test,Tuned-Hybrid,0.075060,0.123349,0.954875,36224,0.0,0.617312
4,1,test,SG-Huber,0.075111,0.126996,0.952167,36224,0.0,NaN
5,1,test,Tuned-Huber-Graph,0.075118,0.126816,0.952302,36224,0.0,0.617312
6,1,test,Huber-Graph,0.075181,0.126826,0.952294,36224,0.0,NaN
7,1,test,Ridge,0.075184,0.122704,0.955345,36224,NaN,NaN
8,1,test,RF,0.075372,0.124537,0.954001,36224,NaN,NaN
9,1,test,SG-Ridge,0.075907,0.126016,0.952902,36224,0.0,NaN



=== STORM EVENTS (99th pct tail MAE) ===


,H,model,tail_MAE,threshold
2,1,Huber,0.400907,3.087695
5,1,Huber-Graph,0.420502,3.087695
8,1,Hybrid,0.406733,3.087695
11,1,RF,0.417664,3.087695
14,1,Ridge,0.399814,3.087695
17,1,SG-Huber,0.421756,3.087695
20,1,SG-Ridge,0.414073,3.087695
23,1,Tuned-Huber-Graph,0.420792,3.087695
26,1,Tuned-Hybrid,0.384047,3.087695
29,1,XGB,0.523144,3.087695
